In [1]:
pip install scipy

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 59.8 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install seaborn

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 24.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install scikit-learn

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 34.2 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 25.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [4]:
import scipy.io as scio
import numpy as np
import matplotlib.pyplot as plt

In [5]:
from keras.models import Sequential
from keras.layers import Dense,Dropout
from keras.layers import LSTM,Input
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import pandas as pd
from keras.models import load_model
import random
random.seed(42)
np.random.seed(42)

2026-05-15 03:15:24.434892: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-15 03:15:26.553008: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [6]:
import random
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    # For CUDA determinism (optional but recommended)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True  # slower but repeatable
    torch.backends.cudnn.benchmark = False     # turn off auto-tuning

In [7]:
!git clone https://github.com/ShihaoCui/FlutterPrediction2.git

Cloning into 'FlutterPrediction2'...
remote: Enumerating objects: 348, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 348 (delta 2), reused 10 (delta 1), pack-reused 336 (from 2)
Receiving objects: 100% (348/348), 3.60 GiB | 3.18 MiB/s, done.
Resolving deltas: 100% (14/14), done.
Updating files: 100% (305/305), done.


# Data processing

In [8]:
import os
from glob import glob
import h5py
import numpy as np

def load_gaf_chunk_v73_fixed(file_path):
    images, labels = [], []
    with h5py.File(file_path, 'r') as f:
        grp = f['chunk']
        N = grp['Image'].shape[0]
        for i in range(N):
            img_ref = grp['Image'][i][0]
            img = f[img_ref][()]
            img = np.transpose(img, (2, 1, 0))  # (H, W, C) → (C, W, H)
            images.append(img)

            label_ref = grp['Label'][i][0]
            label = f[label_ref][()][0][0]
            labels.append(label)

    return np.stack(images), np.array(labels)


def load_all_chunks(split_name='GAF_Train', base_path='./FlutterPrediction2/SplitGAF'):
    files = sorted(glob(os.path.join(base_path, f'{split_name}_Part*.mat')))
    print(f'Found {len(files)} files for {split_name}')

    X_all, y_all = [], []
    for file in files:
        try:
            X, y = load_gaf_chunk_v73_fixed(file)
            X_all.append(X)
            y_all.append(y)
            print(f'✔ Loaded {file}: {X.shape[0]} samples')
        except Exception as e:
            print(f'❌ Error loading {file}: {e}')

    X_all = np.concatenate(X_all, axis=0)
    y_all = np.concatenate(y_all, axis=0)
    print(f'✅ Loaded {X_all.shape[0]} samples for {split_name}')
    return X_all, y_all

In [9]:
X_train, y_train = load_all_chunks('GAF_Train')
X_val,   y_val   = load_all_chunks('GAF_Val')
X_test,  y_test  = load_all_chunks('GAF_Test')

# 转换为 PyTorch 格式：NCHW
X_train = X_train.transpose(0, 3, 1, 2)  # (N, C, H, W)
X_val   = X_val.transpose(0, 3, 1, 2)
X_test  = X_test.transpose(0, 3, 1, 2)

Found 100 files for GAF_Train
✔ Loaded ./FlutterPrediction2/SplitGAF/GAF_Train_Part001.mat: 136 samples
✔ Loaded ./FlutterPrediction2/SplitGAF/GAF_Train_Part002.mat: 136 samples
✔ Loaded ./FlutterPrediction2/SplitGAF/GAF_Train_Part003.mat: 136 samples
✔ Loaded ./FlutterPrediction2/SplitGAF/GAF_Train_Part004.mat: 136 samples
✔ Loaded ./FlutterPrediction2/SplitGAF/GAF_Train_Part005.mat: 136 samples
✔ Loaded ./FlutterPrediction2/SplitGAF/GAF_Train_Part006.mat: 136 samples
✔ Loaded ./FlutterPrediction2/SplitGAF/GAF_Train_Part007.mat: 136 samples
✔ Loaded ./FlutterPrediction2/SplitGAF/GAF_Train_Part008.mat: 136 samples
✔ Loaded ./FlutterPrediction2/SplitGAF/GAF_Train_Part009.mat: 136 samples
✔ Loaded ./FlutterPrediction2/SplitGAF/GAF_Train_Part010.mat: 136 samples
✔ Loaded ./FlutterPrediction2/SplitGAF/GAF_Train_Part011.mat: 136 samples
✔ Loaded ./FlutterPrediction2/SplitGAF/GAF_Train_Part012.mat: 136 samples
✔ Loaded ./FlutterPrediction2/SplitGAF/GAF_Train_Part013.mat: 136 samples
✔ Loaded

In [10]:
X_train.shape

(13600, 7, 64, 64)

In [11]:
y_train.shape

(13600,)

In [12]:
X_test.shape

(2800, 7, 64, 64)

In [13]:
y_test.shape

(2800,)

In [14]:
X_val.shape

(2800, 7, 64, 64)

In [15]:
y_val.shape

(2800,)

In [16]:
import numpy as np
from collections import Counter

def count_labels(y, name="Set"):
    label_counts = Counter(y)
    print(f"\n{name} Label Distribution:")
    for label in sorted(label_counts.keys()):
        print(f"  Label {label}: {label_counts[label]} samples")

# 使用
count_labels(y_train, "Train")
count_labels(y_val, "Val")
count_labels(y_test, "Test")


Train Label Distribution:
  Label 0.0: 2400 samples
  Label 1.0: 4800 samples
  Label 2.0: 6400 samples

Val Label Distribution:
  Label 0.0: 400 samples
  Label 1.0: 1200 samples
  Label 2.0: 1200 samples

Test Label Distribution:
  Label 0.0: 400 samples
  Label 1.0: 800 samples
  Label 2.0: 1600 samples


In [17]:
from sklearn.utils import resample
import numpy as np

def balance_dataset(X, y, per_class, name="Set"):
    X_balanced = []
    y_balanced = []
    
    for label in np.unique(y):
        idx = np.where(y == label)[0]
        X_label = X[idx]
        y_label = y[idx]
        
        if len(idx) < per_class:
            raise ValueError(f"❌ Not enough samples of label {label} in {name} (have {len(idx)}, need {per_class})")
        
        X_sampled, y_sampled = resample(X_label, y_label, 
                                        replace=False, 
                                        n_samples=per_class, 
                                        random_state=42)
        X_balanced.append(X_sampled)
        y_balanced.append(y_sampled)

    X_balanced = np.concatenate(X_balanced, axis=0)
    y_balanced = np.concatenate(y_balanced, axis=0)
    
    print(f"✅ {name} balanced to {len(y_balanced)} samples total ({per_class} per class)")
    return X_balanced, y_balanced

In [18]:
X_train_bal, y_train_bal = balance_dataset(X_train, y_train, per_class=2400, name="Train")
X_val_bal,   y_val_bal   = balance_dataset(X_val,   y_val,   per_class=400,  name="Val")
X_test_bal,  y_test_bal  = balance_dataset(X_test,  y_test,  per_class=400,  name="Test")

✅ Train balanced to 7200 samples total (2400 per class)
✅ Val balanced to 1200 samples total (400 per class)
✅ Test balanced to 1200 samples total (400 per class)


In [19]:
np.savez("GAF_BalancedDatasets.npz", 
         X_train=X_train_bal, y_train=y_train_bal, 
         X_val=X_val_bal, y_val=y_val_bal,
         X_test=X_test_bal, y_test=y_test_bal)

In [22]:
import numpy as np
from sklearn.model_selection import train_test_split

# =========================
# 1. 加载原始 npz 文件
# =========================
data = np.load("GAF_BalancedDatasets.npz")

X_train = data["X_train"]
y_train = data["y_train"]

X_val = data["X_val"]
y_val = data["y_val"]

X_test = data["X_test"]
y_test = data["y_test"]

print("Original shapes:")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:  ", X_val.shape,   "y_val:  ", y_val.shape)
print("X_test: ", X_test.shape,  "y_test: ", y_test.shape)


# =========================
# 2. 合并所有数据
# =========================
X_all = np.concatenate([X_train, X_val, X_test], axis=0)
y_all = np.concatenate([y_train, y_val, y_test], axis=0)

print("\nMerged shapes:")
print("X_all:", X_all.shape)
print("y_all:", y_all.shape)


# =========================
# 3. 重新随机划分数据集
# 比例：70% train, 15% val, 15% test
# =========================

random_seed = 42

# 先分出 test，占总数据的 15%
X_temp, X_test_new, y_temp, y_test_new = train_test_split(
    X_all,
    y_all,
    test_size=0.1,
    random_state=random_seed,
    shuffle=True,
    stratify=y_all
)

# 再从剩下的 85% 中分出 val
# val 占总数据 15%，所以在剩余数据中比例是 0.15 / 0.85
val_ratio = 0.1 / 0.9

X_train_new, X_val_new, y_train_new, y_val_new = train_test_split(
    X_temp,
    y_temp,
    test_size=val_ratio,
    random_state=random_seed,
    shuffle=True,
    stratify=y_temp
)

print("\nNew dataset shapes:")
print("X_train_new:", X_train_new.shape, "y_train_new:", y_train_new.shape)
print("X_val_new:  ", X_val_new.shape,   "y_val_new:  ", y_val_new.shape)
print("X_test_new: ", X_test_new.shape,  "y_test_new: ", y_test_new.shape)


# =========================
# 4. 查看类别分布
# =========================
def show_class_distribution(name, y):
    unique, counts = np.unique(y, return_counts=True)
    print(f"\n{name} class distribution:")
    for u, c in zip(unique, counts):
        print(f"Class {u}: {c}")

show_class_distribution("Train", y_train_new)
show_class_distribution("Val", y_val_new)
show_class_distribution("Test", y_test_new)


# =========================
# 5. 保存新的 npz 文件
# =========================
np.savez(
    "GAF_ReSplitDatasets.npz",
    X_train=X_train_new,
    y_train=y_train_new,
    X_val=X_val_new,
    y_val=y_val_new,
    X_test=X_test_new,
    y_test=y_test_new
)

print("\nSaved new dataset to: GAF_ReSplitDatasets.npz")

Original shapes:
X_train: (7200, 7, 64, 64) y_train: (7200,)
X_val:   (1200, 7, 64, 64) y_val:   (1200,)
X_test:  (1200, 7, 64, 64) y_test:  (1200,)

Merged shapes:
X_all: (9600, 7, 64, 64)
y_all: (9600,)

New dataset shapes:
X_train_new: (7679, 7, 64, 64) y_train_new: (7679,)
X_val_new:   (961, 7, 64, 64) y_val_new:   (961,)
X_test_new:  (960, 7, 64, 64) y_test_new:  (960,)

Train class distribution:
Class 0.0: 2560
Class 1.0: 2560
Class 2.0: 2559

Val class distribution:
Class 0.0: 320
Class 1.0: 320
Class 2.0: 321

Test class distribution:
Class 0.0: 320
Class 1.0: 320
Class 2.0: 320

Saved new dataset to: GAF_ReSplitDatasets.npz
